## Neighbourhood simulation
Where does a sound-driven signal actually go in the real wiring, when the path is not picked in advance?

---

The idea: 03 only ran charge along a hand-picked path, so reaching the end was close to guaranteed
- instead, grow a bounded neighbourhood outward from the JO-B neurons, one hop at a time
- read only the edges touching neurons already reached, never the whole connectome
- then run LIF on that neighbourhood and see which motor neurons actually fire

Why do this at all?
- It is already known, from behaviour, that a fly hears something and moves its legs. This notebook does not try to prove that.
- What it checks is whether the real wiring in this dataset can carry that signal through when it is modelled as simple charge and leak, and how long it takes. Behaviour says what happens at the outside, the simulation checks the mechanism underneath.

Why not reuse the hand-picked path?
- 03 and 04 followed a path chosen in advance, so reaching the end was close to guaranteed, and the test could not surprise anyone.
- Here the signal starts at the hearing neurons and is given all the real nearby connections.
- Where it ends up is then a result, not a setup. It could reach the expected jump/leg motor neurons, unexpected ones, or die out.

Why a neighbourhood and not the whole connectome?
- The full data is ~166k neurons and 151M connections, most of it irrelevant to a sound pulse.
- Growing outward a few hops from the hearing neurons keeps it small enough to run and to read.

Baseline assumed: a healthy fly in normal condition, hears a sound, nothing else competing (no smell, vision or internal state).

Known limits, stated up front:
- the edge weights are unsigned synapse counts, so the model can only excite, never inhibit
- inputs from outside the neighbourhood are missing
- the weight cutoff and hop limit are choices, so they get tested in a sensitivity check

So a result here is a claim about the strong nearby wiring only, not a claim about what the fly does in general.

### Init

In [1]:
import pandas as pd
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.compute as pc
import duckdb
import networkx as nx

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

df = pd.read_feather(neurons_file)
traced = df[df['status'] == 'Traced']
id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()

# lazy handle on the big file, nothing is loaded until a filtered read asks for it
dataset = ds.dataset(connections_file, format="feather")

#### Pulling out the neighbourhood around the hearing neurons

In [ ]:
jo_test_id = 42619  # JO-B1_a, the neuron hand-traced in 04
weight = 3

def query_edges(neuron_ids, min_weight):
    if not isinstance(neuron_ids, (list, set, tuple)):
        neuron_ids = [neuron_ids]
    expr = pc.field("body_pre").isin(neuron_ids) & (pc.field("weight") >= min_weight)
    return dataset.to_table(filter=expr).to_pandas()

edges = query_edges(jo_test_id, weight)
print(edges.shape)
print(edges.sort_values('weight', ascending=False).head())

(60, 3)
   body_pre  body_post  weight
0     42619      15233      46
1     42619     525475      45
2     42619      13014      42
3     42619      11378      42
4     42619      12066      39
